In [17]:
import pandas as pd

fluxes = pd.read_csv('fluxes.csv')
fluxes = fluxes.set_index("Var1")
fluxes.std(axis=1).to_csv('sd.csv')

fluxes.loc['PGIc_f']

Var2_1       0.705746
Var2_2       0.522159
Var2_3       0.820110
Var2_4       0.971271
Var2_5       0.906513
               ...   
Var2_7996    0.452744
Var2_7997    0.074555
Var2_7998    0.317464
Var2_7999    0.361057
Var2_8000    0.391794
Name: PGIc_f, Length: 8000, dtype: float64

In [1]:
def expand_rxn(rxn, label="id", integerize=True):

    def lab(m): return m if label == "obj" else getattr(m, label)

    reactants, products = [], []
    for met, coeff in rxn.metabolites.items():
        n = abs(coeff)
        n_int = int(round(n)) if integerize else 1
        if coeff < 0:
            reactants.extend([lab(met)] * n_int)
        elif coeff > 0:
            products.extend([lab(met)] * n_int)
    return reactants, products

In [2]:
import pandas as pd
from cobra.io import read_sbml_model

chlamymodel = read_sbml_model("photo_model_last.xml")

columns = ['#reaction_ID', 'reactant_IDs(atom)', 'product_IDs(atom)', 'reversibility']
df = pd.DataFrame(
    [[rxn.id, expand_rxn(rxn, label="id")[0], expand_rxn(rxn, label="id")[1], rxn.reversibility]
     for rxn in chlamymodel.reactions],
    columns=columns
)

No objective in listOfObjectives
No objective coefficients in model. Unclear what should be optimized


In [3]:
df.to_csv('metabolites.csv')

In [8]:
import re
import string

MAPPING_FILE = "all_mapping.C.sorted.txt"
OUTPUT_FILE  = "modreactionz.tsv"
UNIQUE       = [c for c in string.ascii_lowercase]
pat         = re.compile(r"(?P<rxn>\S+)\s+(?P<Lspec>\S+):C#(?P<Lidx>\d+)\s*=\s*(?P<Rspec>\S+):C#(?P<Ridx>\d+)")


def pretty(spec, suffix=""):
    base = spec.split("[")[0].replace("_", "")
    if base and base[0].isdigit():
        base = "n" + base
    return f"{base}{spec.split('[')[1].split(']')[0].upper()}{suffix}"

# parse mapping file
by_rxn = {}
with open(MAPPING_FILE) as fh:
    for line in fh:
        m = pat.match(line.strip())
        if not m:
            continue
        rxn = m.group("rxn")
        edge = ((m.group("Lspec"), int(m.group("Lidx"))),
                (m.group("Rspec"), int(m.group("Ridx"))))
        by_rxn.setdefault(rxn, []).append(edge)


FileNotFoundError: [Errno 2] No such file or directory: 'all_mapping.C.sorted.txt'

In [7]:
def build_labels(pairs, left_specs, right_specs):
    """Assign atom-tracking letters to all species occurrences.
    Returns (L_occ, R_occ, L_max, R_max) dicts; occ-dicts are
    sp -> {occurrence_index -> {carbon_index -> letter}}.
    """
    if not pairs:
        return {}, {}, {}, {}

    def count(lst):
        c = {}
        for x in lst: c[x] = c.get(x, 0) + 1
        return c

    left_mult  = count(left_specs)
    right_mult = count(right_specs)

    # group product targets per reactant carbon, sorted for determinism
    prod_targets = {}
    for (Ls, Li), (Rs, Ri) in pairs:
        prod_targets.setdefault((Ls, Li), []).append((Rs, Ri))
    for v in prod_targets.values():
        v.sort()

    letter_iter   = iter(UNIQUE)
    ltrs_for      = {}   # (Ls, Li) -> [letter per left occurrence]
    L_occ         = {}   # sp -> {occ -> {idx -> letter}}

    for (Ls, Li), _ in sorted(prod_targets):
        k    = left_mult.get(Ls, 1)
        ltrs = [next(letter_iter) for _ in range(k)]
        ltrs_for[(Ls, Li)] = ltrs
        for occ, ltr in enumerate(ltrs):
            L_occ.setdefault(Ls, {}).setdefault(occ, {})[Li] = ltr

    R_occ     = {}   # sp -> {occ -> {idx -> letter}}
    r_seen    = {}   # sp -> count of assignments so far

    for (Ls, Li), targets in sorted(prod_targets.items()):
        ltrs   = ltrs_for[(Ls, Li)]
        k_left = len(ltrs)
        for i, (Rs, Ri) in enumerate(targets):
            ltr   = ltrs[i % k_left]
            p_k   = right_mult.get(Rs, 1)
            p_occ = min(r_seen.get(Rs, 0), p_k - 1)
            r_seen[Rs] = r_seen.get(Rs, 0) + 1
            R_occ.setdefault(Rs, {}).setdefault(p_occ, {})[Ri] = ltr

    L_max = {}
    R_max = {}
    for (Ls, Li), (Rs, Ri) in pairs:
        L_max[Ls] = max(L_max.get(Ls, 0), Li)
        R_max[Rs] = max(R_max.get(Rs, 0), Ri)

    return L_occ, R_occ, L_max, R_max


In [20]:
df

,#reaction_ID,reactant_IDs(atom),product_IDs(atom),reversibility
0,RBPCh,"[rb15bp[h], co2[h]]","[3pg[h], 3pg[h]]",False
1,GAPDH_nadp_hi,[3pg[h]],[dhap[h]],False
2,ALDh,"[dhap[h], e4p[h]]",[s17bp[h]],True
3,SBPase,[s17bp[h]],[s7p[h]],False
4,FBAh,"[dhap[h], dhap[h]]",[fdp_B[h]],True
...,...,...,...,...
69,GLUtm,[glu_L[c]],[glu_L[m]],True
70,ALAtm,[ala_L[c]],[ala_L[m]],True
71,PEPtm,[pep[c]],[pep[m]],True
72,CO2tm,[co2[m]],[co2[c]],True


In [19]:
def build_modreactions(df):
    rows = ["\t".join(["#reaction.ID", "reactant.IDs(atom)", "product.IDs(atom)", "reversibility"])]

    def parse_side(text):
        return [f"{m}[{c}]" for m, c in species_pat.findall(str(text))]

    def letters(sp, occ, occ_dict, max_dict):
        idxmap = occ_dict.get(sp, {}).get(occ, {})
        return "".join(filter(None, (idxmap.get(i, "") for i in range(1, max_dict.get(sp, 0) + 1))))

    def assemble(specs, L_occ, max_dict, suffix=""):
        seen, parts = {}, []
        for sp in specs:
            occ      = seen.get(sp, 0)
            seen[sp] = occ + 1
            ltr      = letters(sp, occ, L_occ, max_dict)
            nm       = pretty(sp, suffix=suffix)
            parts.append(f"{nm}({ltr})" if ltr else nm)
        return " + ".join(parts)

    for _, r in df.iterrows():
        rxn         = r["#reaction_ID"]
        left_specs  = parse_side(r["reactant_IDs(atom)"])
        right_specs = parse_side(r["product_IDs(atom)"])

        Locc, Rocc, Lmax, Rmax = build_labels(by_rxn.get(rxn, []), left_specs, right_specs)

        left_str  = assemble(left_specs,  Locc, Lmax)
        right_str = assemble(right_specs, Rocc, Rmax)

        if str(rxn).lower() == "biomass":
            right_str = "biomass"
        elif not left_str and right_str:   # sink
            left_str  = assemble(right_specs, Rocc, Rmax, suffix=".ex")
        elif left_str and not right_str:   # source
            right_str = assemble(left_specs,  Locc, Lmax, suffix=".ex")

        rev_raw = str(r["reversibility"]).strip().lower()
        rev = "1" if rev_raw in {"true", "1", "yes"} else "0"
        rows.append("\t".join([rxn, left_str, right_str, rev]))

    with open(OUTPUT_FILE, "w") as fh:
        fh.write("\n".join(rows))
    print(f"Wrote {OUTPUT_FILE} with {len(df)} reactions ({len(rows) - 1} rows written).")


build_modreactions(df)


Wrote modreactionz.tsv with 74 reactions (74 rows written).
